# NeuroDB SQL Playground

Use this notebook when asked a new question during the demonstration. Edit the SQL in the next cell and run that cell. Queries use a separate read only MySQL account, stop after five seconds, and show at most 200 rows. Use one statement at a time.


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd() if (Path.cwd() / 'neurodb').is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from neurodb.server import start, ensure_reader
from neurodb.custom_sql import dataframe as query

start()
ensure_reader()
display(query('SELECT CURRENT_USER() AS sql_user, DATABASE() AS database_name'))

MySQL 8.4.11 ready


,sql_user,database_name
0,neurodb_reader@localhost,neurodb


In [2]:
query("""
SELECT *
FROM trial_context
LIMIT 10
""")

,participant_id,bids_label,design_variant,task_order,analytic_inclusion_status,run_id,run_type,run_number,behavior_run_number,trial_id,...,chosen_ship,visited_planet,visited_pad,reward,reward_magnitude,transition_type,reward_condition,contingency_condition,state_condition,response_time
0,1,sub-MBMF000202,mixed,2,unknown,1,structural,1,1,1,...,1,2,3,0,0.0,rare,1,1,1,1.404264
1,1,sub-MBMF000202,mixed,2,unknown,1,structural,1,1,2,...,2,2,4,1,71.0,common,1,1,1,0.367155
2,1,sub-MBMF000202,mixed,2,unknown,1,structural,1,1,3,...,2,2,3,1,66.0,common,1,1,1,0.252591
3,1,sub-MBMF000202,mixed,2,unknown,1,structural,1,1,4,...,1,1,1,0,0.0,common,1,1,1,0.271668
4,1,sub-MBMF000202,mixed,2,unknown,1,structural,1,1,5,...,1,1,1,0,0.0,common,1,1,1,0.198552
5,1,sub-MBMF000202,mixed,2,unknown,1,structural,1,1,6,...,1,2,3,0,0.0,rare,1,1,1,0.336557
6,1,sub-MBMF000202,mixed,2,unknown,1,structural,1,1,7,...,2,2,4,1,94.0,common,1,1,1,0.246467
7,1,sub-MBMF000202,mixed,2,unknown,1,structural,1,1,8,...,2,1,1,0,0.0,rare,1,1,1,0.277233
8,1,sub-MBMF000202,mixed,2,unknown,1,structural,1,1,9,...,2,2,4,0,0.0,common,1,1,1,0.298245
9,1,sub-MBMF000202,mixed,2,unknown,1,structural,1,1,10,...,1,1,1,0,0.0,common,2,1,1,0.271866


## Inspect a plan

The example below finds a functional run ID from the current database, then explains a 60 second event window. Replace the SELECT statement to inspect a different retrieval query.


In [3]:
example_run = int(query("SELECT run_id FROM runs WHERE run_type='functional' ORDER BY run_id LIMIT 1").iloc[0, 0])
print('Functional run ID', example_run)
print(query("""
EXPLAIN ANALYZE
SELECT event_id, onset_seconds
FROM events
WHERE run_id = %s AND onset_seconds < 60
ORDER BY onset_seconds, event_id
""", (example_run,)).iloc[0, 0])

Functional run ID 2
-> Filter: ((`events`.run_id = 2) and (`events`.onset_seconds < 60))  (cost=9.1 rows=40) (actual time=0.0136..0.0176 rows=40 loops=1)
    -> Covering index range scan on events using idx_events_run_onset over (run_id = 2 AND onset_seconds < 60)  (cost=9.1 rows=40) (actual time=0.0126..0.0147 rows=40 loops=1)



## Quick reference

| Need | Start with |
| --- | --- |
| Behavioral choices, reward and response time | `trial_context` view |
| Runs and scan metadata | `runs` |
| Trial identity and missing responses | `trials` |
| Event timing | `events` joined to `event_types` |
| Optional model estimates | `latent_values`, `variable_definitions`, `participant_fits` |

For structure, run `query('SHOW TABLES')`, `query('DESCRIBE trials')`, or `query('SHOW INDEX FROM events')`. The view `trial_context` joins the core behavioral tables; it is a regular view, not a materialized table.
